# drawdown_rebound_reward_train_v1

Runtime notebook for an upstream reward/model-selection diagnostic after fixed-actor execution governors failed to reach `relative_return >= 30pp` and `max_dd <= 15%` without underparticipating.

## Governance

- **Direction**: `reward` with `model_selection` evidence.
- **Canonical baseline**: `balanced_rule_residual_v22` remains production/canonical context.
- **Direct comparison baseline**: `risk_budget_usage_v1`, because this profile keeps the same two-speed budget-usage architecture and changes the training reward/selection emphasis.
- **Role**: `diagnose` / `ablate`; no runtime binding from this run.
- **Changed layer**: `reward`, `selection`, and short `sampling`; no execution overlay is enabled.
- **Target window**: fixed split `test_proxy`, slices `test_2020`, `test_2022`, `test_2023_2024`, plus saved walk-forward workers `wf_2020`, `wf_2022`, `wf_2023_2024`.
- **Accepted trade-off**: keep only if the model improves path risk without collapsing rebound/trend participation; target remains `relative_return >= 30pp` and `max_dd <= 15%`.
- **Hypothesis**: moving drawdown and rebound terms upstream into the actor objective can avoid the post-hoc cap failure mode. The profile combines path drawdown penalty, terminal CVaR, rebound opportunity cost, and mode-conditional rewards inside the existing risk-budget action mapping.
- **Failure mode**: if later checkpoints or walk-forward windows show lower validation but higher OOS drawdown/underparticipation, reward shaping is still overfit or too weak for this actor family.
- **Comparison-table row key**: `drawdown_rebound_reward_train_v1`

## Historical duplicate check

- **Closest success**: `risk_budget_usage_v1` showed two-speed budget usage can lower 2022 exposure compared with high-return residual actors.
- **Closest failure**: `balanced_rule_residual_v22_continued_retention_v1` found that continuing from v22 widened risk; phase-balanced sampling alone also widened risk rather than fixing style.
- **Why this is not a duplicate**: this run does not continue v22 and does not add another post-hoc cap. It keeps the risk-budget mapping but changes the upstream reward/selection terms and verifies the terminal CVaR path is active for `use_moderate_reward`.

## External references used

- Qlib repository: https://github.com/microsoft/qlib
- Qlib paper: https://arxiv.org/abs/2009.11189
- Deep Momentum Networks: https://arxiv.org/abs/1904.04912
- DeepTrader / drawdown-aware RL precedent: https://ojs.aaai.org/index.php/AAAI/article/view/16144
- Probability of Backtest Overfitting: https://papers.ssrn.com/sol3/papers.cfm?abstract_id=2326253


In [1]:
from __future__ import annotations

import importlib.util
import json
from pathlib import Path
import subprocess
import sys
from typing import Any

from IPython.display import Markdown, display
import pandas as pd


def find_repo_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / 'AGENTS.md').exists() and (path / 'src' / 'soxl_quant').exists():
            return path
    raise RuntimeError(f'could not locate repo root from {start}')


ROOT = find_repo_root(Path.cwd())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

CARD = 'drawdown_rebound_reward_train_v1'
CARD_DIR = ROOT / 'runtime' / 'reward' / 'card' / CARD
PROFILE_PATH = CARD_DIR / 'profile.yaml'
MODEL_CARD_DIR = ROOT / 'runtime' / 'model' / 'card' / CARD
LOG_PATH = CARD_DIR / 'train_walkforward.log'
TARGET_RELATIVE_RETURN = 30.0
TARGET_MAX_DD = 15.0
SEED = 20260524

assert PROFILE_PATH.exists(), PROFILE_PATH
print(f'card_dir={CARD_DIR.relative_to(ROOT)}')
print(f'profile={PROFILE_PATH.relative_to(ROOT)}')


card_dir=runtime/reward/card/drawdown_rebound_reward_train_v1
profile=runtime/reward/card/drawdown_rebound_reward_train_v1/profile.yaml


## Harness patch check

This experiment depends on terminal CVaR being active in the `use_moderate_reward` branch. The repository test below is intentionally called from the notebook so the runtime artifact records that the training objective is actually wired.

In [2]:
spec = importlib.util.spec_from_file_location('test_reward', ROOT / 'tests' / 'test_reward.py')
module = importlib.util.module_from_spec(spec)
assert spec.loader is not None
spec.loader.exec_module(module)
module.test_moderate_reward_applies_terminal_cvar_penalty()
print('moderate terminal CVaR reward path: verified')


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


moderate terminal CVaR reward path: verified


## Train + walk-forward command

This is deliberately short (`max_iters=8`, `stage_iters=4`) to test direction and checkpoint behavior before committing to a long PPO branch. It still runs the saved walk-forward workers so overfit and underparticipation show up immediately.

In [3]:
expected_summary = MODEL_CARD_DIR / f'soxl_{CARD}_summary.json'
force_train = False

cmd = [
    str(ROOT / 'qlib_venv' / 'bin' / 'python'),
    str(ROOT / 'soxl_qlib_strategy_v2.py'),
    '--profiles', CARD,
    '--skip-data-refresh',
    '--seed', str(SEED),
    '--walk-forward',
]
env = {'PYTHONPATH': 'src:.', **dict(__import__('os').environ)}
print(' '.join(str(part) for part in cmd))
if expected_summary.exists() and LOG_PATH.exists() and not force_train:
    print(f'reusing existing artifacts: {expected_summary.relative_to(ROOT)}')
else:
    proc = subprocess.run(cmd, cwd=ROOT, env=env, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    LOG_PATH.write_text(proc.stdout, encoding='utf-8')
    print(proc.stdout[-5000:])
    if proc.returncode != 0:
        raise RuntimeError(f'training command failed with exit={proc.returncode}; see {LOG_PATH.relative_to(ROOT)}')
print(f'log={LOG_PATH.relative_to(ROOT)}')


/Users/rziiiii/.openclaw/workspace/qlib_venv/bin/python /Users/rziiiii/.openclaw/workspace/soxl_qlib_strategy_v2.py --profiles drawdown_rebound_reward_train_v1 --skip-data-refresh --seed 20260524 --walk-forward
reusing existing artifacts: runtime/model/card/drawdown_rebound_reward_train_v1/soxl_drawdown_rebound_reward_train_v1_summary.json
log=runtime/reward/card/drawdown_rebound_reward_train_v1/train_walkforward.log


In [4]:
def load_json(path: Path) -> dict[str, Any]:
    with path.open() as f:
        return json.load(f)


def metric_row(label: str, payload: dict[str, Any], *, kind: str) -> dict[str, Any]:
    test_proxy = payload.get('test_proxy', payload)
    return {
        'label': label,
        'kind': kind,
        'return': float(test_proxy.get('return', payload.get('return', float('nan')))),
        'sharpe': float(test_proxy.get('sharpe', payload.get('sharpe', float('nan')))),
        'max_dd': float(test_proxy.get('max_dd', payload.get('max_dd', float('nan')))),
        'relative_return': float(test_proxy.get('relative_return', payload.get('relative_return', float('nan')))),
        'buy_and_hold_return': float(test_proxy.get('buy_and_hold_return', payload.get('buy_and_hold_return', float('nan')))),
        'avg_pos': float(test_proxy.get('avg_pos', float('nan'))),
        'trades': int(test_proxy.get('trades', payload.get('trades', 0))),
        'target_pass': bool(
            float(test_proxy.get('relative_return', payload.get('relative_return', float('-inf')))) >= TARGET_RELATIVE_RETURN
            and float(test_proxy.get('max_dd', payload.get('max_dd', float('inf')))) <= TARGET_MAX_DD
        ),
    }

summary_path = MODEL_CARD_DIR / f'soxl_{CARD}_summary.json'
if not summary_path.exists():
    raise FileNotFoundError(summary_path)
summary = load_json(summary_path)
rows = [metric_row('main', summary, kind='fixed_split')]

# Baselines for comparison; missing rows are skipped rather than inferred.
for baseline_name in ['risk_budget_usage_v1', 'balanced_rule_residual_v22']:
    baseline_path = ROOT / 'runtime' / 'model' / 'card' / baseline_name / f'soxl_{baseline_name}_summary.json'
    if baseline_path.exists():
        rows.append(metric_row(baseline_name, load_json(baseline_path), kind='baseline'))

wf_dir = ROOT / 'runtime' / 'soxl' / 'wf_workers'
wf_payloads = {}
for wf_name in ['wf_2020', 'wf_2022', 'wf_2023_2024']:
    path = wf_dir / f'{CARD}-{wf_name}-seed{SEED}.json'
    if path.exists():
        payload = load_json(path)
        wf_payloads[wf_name] = payload
        rows.append(metric_row(wf_name, payload, kind='walk_forward'))
    else:
        rows.append({'label': wf_name, 'kind': 'walk_forward', 'missing': True})

metrics = pd.DataFrame(rows)
metrics.to_csv(CARD_DIR / 'metrics_summary.csv', index=False)
display(metrics)


,label,kind,return,sharpe,max_dd,relative_return,buy_and_hold_return,avg_pos,trades,target_pass
0,main,fixed_split,392.580698,1.026548,37.676486,-2026.444285,2542.027442,0.281184,0,False
1,risk_budget_usage_v1,baseline,77.541789,0.646808,18.860762,-424.056726,501.598514,0.140906,0,False
2,balanced_rule_residual_v22,baseline,695.451601,0.941247,54.517981,NaN,NaN,0.508523,0,False
3,wf_2020,walk_forward,53.606145,1.576978,20.934520,-297.064031,355.736160,0.327062,37,False
4,wf_2022,walk_forward,-7.000281,-0.657429,9.875679,53.563568,-76.614257,0.165855,22,True
5,wf_2023_2024,walk_forward,-2.958779,-0.370753,9.737972,28.876180,-39.952710,0.170288,23,False


In [5]:
history = summary.get('selection_history', []) or summary.get('selection', {}).get('history', [])
history_df = pd.DataFrame(history)
if not history_df.empty:
    history_df.to_csv(CARD_DIR / 'selection_history.csv', index=False)
    display(history_df[['iter', 'score', 'return', 'sharpe', 'max_dd', 'avg_pos'] if 'avg_pos' in history_df.columns else ['iter', 'score', 'return', 'sharpe', 'max_dd']])
else:
    print('no selection history found')

slices = summary.get('slices', {})
slice_rows = []
for name, row in slices.items():
    out = {'slice': name}
    out.update(row)
    slice_rows.append(out)
slices_df = pd.DataFrame(slice_rows)
if not slices_df.empty:
    slices_df.to_csv(CARD_DIR / 'slice_metrics.csv', index=False)
    display(slices_df)


,iter,score,return,sharpe,max_dd,avg_pos
0,0,-9.063366,1.055316,0.207664,27.030917,0.362588
1,4,14.289034,13.001993,0.690098,23.546830,0.349317
2,8,11.237874,7.205514,0.577706,13.168937,0.213089


,slice,return,sharpe,max_dd,avg_pos,stress_avg_pos,stress_days,buy_and_hold_return,relative_return
0,test_2020,58.672129,1.869642,20.454723,0.327062,0.149324,37.0,355.736160,-297.064031
1,test_2022,-31.944356,-1.409414,35.878546,0.185465,0.132267,86.0,-76.614257,55.990314
2,test_2023_2024,52.300504,1.104254,19.908058,0.273526,0.168421,76.0,55.654474,-3.353969


In [6]:
main_row = metrics[(metrics['label'] == 'main') & (metrics['kind'] == 'fixed_split')].iloc[0].to_dict()
wf_rows = metrics[metrics['kind'] == 'walk_forward'].copy()
wf_missing = wf_rows['missing'].fillna(False).astype(bool) if 'missing' in wf_rows else pd.Series(False, index=wf_rows.index)
wf_target_pass = wf_rows['target_pass'].fillna(False).astype(bool) if 'target_pass' in wf_rows else pd.Series(False, index=wf_rows.index)
wf_complete = len(wf_rows) == 3 and int(wf_missing.sum()) == 0
wf_target_pass_count = int(wf_target_pass.sum()) if wf_complete else 0
all_target_pass = bool(main_row['target_pass'] and wf_complete and wf_target_pass_count == 3)

if all_target_pass:
    decision = 'candidate_needs_longer_confirmation'
elif history_df.empty:
    decision = 'diagnostic_incomplete_missing_history'
elif float(main_row['max_dd']) > TARGET_MAX_DD:
    decision = 'diagnostic_no_promotion_drawdown_still_too_high'
elif float(main_row['relative_return']) < TARGET_RELATIVE_RETURN:
    decision = 'diagnostic_no_promotion_underparticipation'
else:
    decision = 'diagnostic_no_promotion_walkforward_blocked'

selection = {
    'card': CARD,
    'decision': decision,
    'target': {'relative_return_min_pp': TARGET_RELATIVE_RETURN, 'max_dd_max_pct': TARGET_MAX_DD},
    'main': main_row,
    'walk_forward_complete': wf_complete,
    'walk_forward_target_pass_count': wf_target_pass_count,
    'selection_history': history,
    'artifacts': {
        'notebook': str((CARD_DIR / f'{CARD}.ipynb').relative_to(ROOT)),
        'profile': str(PROFILE_PATH.relative_to(ROOT)),
        'train_log': str(LOG_PATH.relative_to(ROOT)),
        'summary': str(summary_path.relative_to(ROOT)),
        'metrics_summary': str((CARD_DIR / 'metrics_summary.csv').relative_to(ROOT)),
        'selection_history': str((CARD_DIR / 'selection_history.csv').relative_to(ROOT)),
        'slice_metrics': str((CARD_DIR / 'slice_metrics.csv').relative_to(ROOT)),
    },
    'interpretation': [],
}

if not bool(main_row['target_pass']):
    if float(main_row['max_dd']) > TARGET_MAX_DD:
        selection['interpretation'].append('Main fixed-split drawdown remains above 15%, so the reward objective is not promotable.')
    if float(main_row['relative_return']) < TARGET_RELATIVE_RETURN:
        selection['interpretation'].append('Main fixed-split relative return is below +30pp, indicating underparticipation or weak alpha capture.')
if wf_complete and wf_target_pass_count < 3:
    selection['interpretation'].append('Saved walk-forward workers do not pass all target windows; no runtime binding.')
if history:
    best_iter = max(history, key=lambda row: float(row.get('score', float('-inf')))).get('iter')
    selection['interpretation'].append(f'Best validation checkpoint selected at iter {best_iter}; inspect later-stage drift before extending training.')
selection['interpretation'].append('Next step should compare retained checkpoints and poor windows before any longer training branch.')

def json_safe(value: Any) -> Any:
    if isinstance(value, dict):
        return {str(key): json_safe(item) for key, item in value.items()}
    if isinstance(value, list):
        return [json_safe(item) for item in value]
    if hasattr(value, 'item') and not isinstance(value, (str, bytes, bool)):
        try:
            value = value.item()
        except (TypeError, ValueError):
            pass
    if pd.isna(value):
        return None
    if isinstance(value, (str, int, float, bool)) or value is None:
        return value
    return str(value)

with (CARD_DIR / 'selection.json').open('w') as f:
    json.dump(json_safe(selection), f, indent=2, allow_nan=False)

lines = [
    f'# {CARD} conclusion',
    '',
    f'- Decision: `{decision}`',
    f'- Main relative return: `{main_row["relative_return"]:.2f}pp`',
    f'- Main MaxDD: `{main_row["max_dd"]:.2f}%`',
    f'- Main target pass: `{bool(main_row["target_pass"])}`',
    f'- Walk-forward complete: `{wf_complete}`',
    f'- Walk-forward target-pass count: `{wf_target_pass_count}/3`',
    '',
    '## Interpretation',
]
lines.extend(f'- {item}' for item in selection['interpretation'])
conclusion_md = '\n'.join(lines)
display(Markdown(conclusion_md))


# drawdown_rebound_reward_train_v1 conclusion

- Decision: `diagnostic_no_promotion_drawdown_still_too_high`
- Main relative return: `-2026.44pp`
- Main MaxDD: `37.68%`
- Main target pass: `False`
- Walk-forward complete: `True`
- Walk-forward target-pass count: `1/3`

## Interpretation
- Main fixed-split drawdown remains above 15%, so the reward objective is not promotable.
- Main fixed-split relative return is below +30pp, indicating underparticipation or weak alpha capture.
- Saved walk-forward workers do not pass all target windows; no runtime binding.
- Best validation checkpoint selected at iter 4; inspect later-stage drift before extending training.
- Next step should compare retained checkpoints and poor windows before any longer training branch.